In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from functools import reduce
import re

### EMAIL FEATURES

In [3]:
email = pd.read_parquet("cert_dataset/email.parquet")
email.head()

,id,date,user,pc,to,cc,bcc,from,activity,size,attachments,content
0,{I1O2-B4EB49RW-7379WSQW},01/02/2010 06:36:41,HDB1666,PC-6793,Louis.Bernard.Garza@dtaa.com,Emery.Ali.Holloway@dtaa.com,Hector.Donovan.Bray@dtaa.com,Hector.Donovan.Bray@dtaa.com,Send,45659,None,"Now Sylvia, the object of Aminta's desire, arr..."
1,{L7E7-V4UX89RR-3036ZDHU},01/02/2010 06:40:02,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Luke.Grant.Mcmahon@dtaa.com,View,34142,None,"On May 14, they picked up 44 more Iroquois at ..."
2,{S8C2-Q8YX87DJ-0516SIWZ},01/02/2010 06:42:48,HDB1666,PC-6793,Quintessa.O.Farrell@harris.com,Hector.Donovan.Bray@dtaa.com,None,Hector.Donovan.Bray@dtaa.com,Send,1310925,C:\28X79b6\0PAGXTJ8.doc(1119253);C:\11b38g6\5M...,Sylvia is notable for its mythological Arcadia...
3,{A1V9-O5BL46SW-1708NAEC},01/02/2010 06:45:42,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Travis.Ezra.Warner@dtaa.com,View,23043,None,Lanctot (1967) and Smith do not identify any s...
4,{N6R0-M2EI82DM-5583LSUM},01/02/2010 06:47:07,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Kenyon.William.Delacruz@dtaa.com,View,25210,None,Much like all the other large groups of prison...


In [5]:
suspicious_keywords = [
    'bypass', 'tunnel', 'unauthorized', 'revenge', 'disappointed', 
    'competitor', 'recruit', 'export', 'downloading', 'illegal',
    'access denied', 'vulnerability', 'exploit', 'vpn', 'cloud', 
    'attachment', 'personal', 'fired', 'quit', 'unfair'
]

pattern = '|'.join(suspicious_keywords)

email['has_suspicious_content'] =(email['content'].str.lower().str.contains(pattern,na=False).astype(int))  

In [6]:
email['date'] = pd.to_datetime(email['date'], format='%m/%d/%Y %H:%M:%S')
email = email.drop(columns=['id'])
email = email.sort_values(by='date')
email.head()

,date,user,pc,to,cc,bcc,from,activity,size,attachments,content,has_suspicious_content
0,2010-01-02 06:36:41,HDB1666,PC-6793,Louis.Bernard.Garza@dtaa.com,Emery.Ali.Holloway@dtaa.com,Hector.Donovan.Bray@dtaa.com,Hector.Donovan.Bray@dtaa.com,Send,45659,None,"Now Sylvia, the object of Aminta's desire, arr...",0
1,2010-01-02 06:40:02,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Luke.Grant.Mcmahon@dtaa.com,View,34142,None,"On May 14, they picked up 44 more Iroquois at ...",0
2,2010-01-02 06:42:48,HDB1666,PC-6793,Quintessa.O.Farrell@harris.com,Hector.Donovan.Bray@dtaa.com,None,Hector.Donovan.Bray@dtaa.com,Send,1310925,C:\28X79b6\0PAGXTJ8.doc(1119253);C:\11b38g6\5M...,Sylvia is notable for its mythological Arcadia...,0
3,2010-01-02 06:45:42,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Travis.Ezra.Warner@dtaa.com,View,23043,None,Lanctot (1967) and Smith do not identify any s...,0
4,2010-01-02 06:47:07,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Kenyon.William.Delacruz@dtaa.com,View,25210,None,Much like all the other large groups of prison...,0


In [7]:
domain = '@dtaa.com'
email['is_external'] = (
    (~email['from'].str.contains(domain, na=False)) |
    (~email['to'].str.contains(domain, na=False))
)
email['has_bcc'] = email['bcc'].notna() & (email['bcc'] != 'None')

email['has_attachment'] = email['attachments'].notna() & (email['attachments'] != 'None')

email['is_external_with_attachment'] = email['is_external'] & email['has_attachment']

email['week'] = email['date'].dt.to_period('W')

email['is_after_hours'] = ( (email['date'].dt.hour > 19) | (email['date'].dt.hour < 7))

email['is_weekend'] = email['date'].dt.dayofweek >= 5

email['is_weekend_or_after_hours'] = email['is_weekend'] | email['is_after_hours']

email['nb_attachments'] = email['attachments'].apply(
    lambda x : len(x.split(';')) if pd.notna(x) and x != None else 0
)

email.head()

,date,user,pc,to,cc,bcc,from,activity,size,attachments,...,has_suspicious_content,is_external,has_bcc,has_attachment,is_external_with_attachment,week,is_after_hours,is_weekend,is_weekend_or_after_hours,nb_attachments
0,2010-01-02 06:36:41,HDB1666,PC-6793,Louis.Bernard.Garza@dtaa.com,Emery.Ali.Holloway@dtaa.com,Hector.Donovan.Bray@dtaa.com,Hector.Donovan.Bray@dtaa.com,Send,45659,None,...,0,False,True,False,False,2009-12-28/2010-01-03,True,True,True,0
1,2010-01-02 06:40:02,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Luke.Grant.Mcmahon@dtaa.com,View,34142,None,...,0,False,False,False,False,2009-12-28/2010-01-03,True,True,True,0
2,2010-01-02 06:42:48,HDB1666,PC-6793,Quintessa.O.Farrell@harris.com,Hector.Donovan.Bray@dtaa.com,None,Hector.Donovan.Bray@dtaa.com,Send,1310925,C:\28X79b6\0PAGXTJ8.doc(1119253);C:\11b38g6\5M...,...,0,True,False,True,True,2009-12-28/2010-01-03,True,True,True,2
3,2010-01-02 06:45:42,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Travis.Ezra.Warner@dtaa.com,View,23043,None,...,0,False,False,False,False,2009-12-28/2010-01-03,True,True,True,0
4,2010-01-02 06:47:07,HDB1666,PC-6793,Hector.Donovan.Bray@dtaa.com,None,None,Kenyon.William.Delacruz@dtaa.com,View,25210,None,...,0,False,False,False,False,2009-12-28/2010-01-03,True,True,True,0


In [23]:
features_email = email.groupby('user').agg(
    nb_emails = ('date','count'),
    nb_emails_sent = ('activity',lambda x : (x=='Send').sum()),
    nb_emails_viewed = ('activity',lambda x : (x=='View').sum()),
    nb_external = ('is_external','sum'),
    nb_bcc = ('has_bcc','sum'),
    nb_external_with_attachment = ('is_external_with_attachment','sum'),
    nb_after_hours_or_weekend = ('is_weekend_or_after_hours','sum'),
    nb_emails_has_suspicious_content = ('has_suspicious_content','sum')
).reset_index()

features_email['email_external_ratio'] = (
    features_email['nb_external'] / features_email['nb_emails']
)

features_email['suspicious_content_ratio'] = (
    features_email['nb_emails_has_suspicious_content'] / features_email['nb_emails']
)

features_email['bcc_email_ratio'] = (
    features_email['nb_bcc'] / features_email['nb_emails']
)
features_email['external_email_with_attachment_ratio'] = (
    features_email['nb_external_with_attachment'] / features_email['nb_emails']
)
features_email['after_hours_or_weekend_ratio'] = (
    features_email['nb_after_hours_or_weekend'] / features_email['nb_emails']
)

emails_per_week = email.groupby(['user', 'week']).size().reset_index(name='nb_emails_week')

weekly_email_activity = email.groupby(['user', 'week']).agg(
    nb_emails = ('activity','count'),
    nb_emails_sent = ('activity',lambda x : (x=='Send').sum()) 
).reset_index()

email_baseline = weekly_email_activity.groupby('user').agg(
    baseline_median_emails = ('nb_emails','median'),
    baseline_median_emails_sent = ('nb_emails_sent','median'),
    baseline_std_emails = ('nb_emails','std'),
    baseline_std_emails_sent = ('nb_emails_sent','std'),
)

weekly_email_activity = weekly_email_activity.merge(email_baseline,on='user',how='left')

weekly_email_activity['zscore_emails'] = (
    (weekly_email_activity['nb_emails'] - weekly_email_activity['baseline_median_emails']) / weekly_email_activity['baseline_std_emails'].clip(lower=1)
)

weekly_email_activity['zscore_emails_sent'] = (
    (weekly_email_activity['nb_emails_sent'] - weekly_email_activity['baseline_median_emails_sent']) / weekly_email_activity['baseline_std_emails_sent'].clip(lower=1)
)

zscore_email_features = weekly_email_activity.groupby('user').agg(
    max_zscore_emails      = ('zscore_emails', 'max'),
    max_zscore_emails_sent = ('zscore_emails_sent', 'max'),
).reset_index()

features_email = features_email.merge(zscore_email_features, on='user', how='left')

avg_emails_per_week = emails_per_week.groupby('user')['nb_emails_week'].mean().reset_index()

avg_emails_per_week.columns = ['user', 'avg_emails_per_week']

features_email = features_email.merge(avg_emails_per_week, on='user', how='left')

sent_emails = email[email['activity'] == 'Send']

features_sent = sent_emails.groupby('user').agg(
    avg_size_sent_email            = ('size', 'mean'),  
).reset_index()
features_email = features_email.merge(features_sent, on='user', how='left')

features_email.head() 

,user,nb_emails,nb_emails_sent,nb_emails_viewed,nb_external,nb_bcc,nb_external_with_attachment,nb_after_hours_or_weekend,nb_emails_has_suspicious_content,email_external_ratio,suspicious_content_ratio,bcc_email_ratio,external_email_with_attachment_ratio,after_hours_or_weekend_ratio,max_zscore_emails,max_zscore_emails_sent,avg_emails_per_week,avg_size_sent_email
0,AAB0162,3146,1122,2024,967,0,161,0,220,0.307374,0.069930,0.0,0.051176,0.000000,0.0,1.843913,42.513514,1.994349e+05
1,AAB0398,3811,1348,2463,1462,0,450,215,289,0.383626,0.075833,0.0,0.118079,0.056416,0.0,2.314657,51.500000,3.568791e+05
2,AAC0610,1052,388,664,382,0,139,0,93,0.363118,0.088403,0.0,0.132129,0.000000,0.0,2.661585,14.216216,8.273392e+05
3,AAC0668,3115,1093,2022,1092,0,219,0,331,0.350562,0.106260,0.0,0.070305,0.000000,0.0,2.145350,42.094595,4.068177e+05
4,AAC3270,344,120,224,89,0,48,0,25,0.258721,0.072674,0.0,0.139535,0.000000,0.0,2.951763,4.648649,1.333953e+06


In [9]:
logon = pd.read_parquet("cert_dataset/logon.parquet")
logon.head()

,id,date,user,pc,activity
0,{F3X8-Y2GT43DR-4906OHBL},01/02/2010 02:19:18,DNS1758,PC-0414,Logon
1,{B4Q0-D0GM24KN-3704MAII},01/02/2010 02:31:12,DNS1758,PC-0414,Logoff
2,{T7J1-D4HK34KV-5476TCIJ},01/02/2010 02:34:02,DNS1758,PC-5313,Logon
3,{S4Y6-D8MQ05SA-0759HLIS},01/02/2010 02:53:30,DNS1758,PC-5313,Logoff
4,{F3P0-E7FH78CV-4874FRGZ},01/02/2010 04:07:31,DNS1758,PC-0012,Logon


In [10]:
logon['date'] = pd.to_datetime(logon['date'], format='%m/%d/%Y %H:%M:%S')
logon = logon.drop(columns=['id'])
logon = logon.sort_values(by='date')
logon.head()

,date,user,pc,activity
0,2010-01-02 02:19:18,DNS1758,PC-0414,Logon
1,2010-01-02 02:31:12,DNS1758,PC-0414,Logoff
2,2010-01-02 02:34:02,DNS1758,PC-5313,Logon
3,2010-01-02 02:53:30,DNS1758,PC-5313,Logoff
4,2010-01-02 04:07:31,DNS1758,PC-0012,Logon


In [26]:
logon['is_after_hours'] = ( (logon['date'].dt.hour > 19) | (logon['date'].dt.hour < 7))
logon['week'] = logon['date'].dt.to_period('W')

logon_events = logon[logon['activity'] == 'Logon'].sort_values(by=['user','date']).copy()
logoff_events = logon[logon['activity'] == 'Logoff'].sort_values(by=['user','date']).copy()

logon_events['rank'] = logon_events.groupby('user').cumcount()
logoff_events['rank'] = logoff_events.groupby('user').cumcount()

sessions = pd.merge(
    logon_events[['user','rank','pc','date']],
    logoff_events[['user','rank','date']],
    on=['user','rank'],
    suffixes=['_start','_end']
)

sessions['duration_hours'] = (
    sessions['date_end'] - sessions['date_start']
).dt.total_seconds() / 3600

sessions = sessions[(sessions['duration_hours'] > 0) & (sessions['duration_hours'] < 24)]


In [12]:
features_sessions = sessions.groupby('user').agg(
    average_session_duration     = ('duration_hours', 'mean'),
    session_duration_variability = ('duration_hours', 'std'),
    nb_sessions                  = ('duration_hours', 'count'),
    nb_short_sessions            = ('duration_hours', lambda x: (x < 0.1).sum()),
).reset_index()

features_sessions['ratio_short_sessions'] = (
    features_sessions['nb_short_sessions'] / features_sessions['nb_sessions']
)

logon_events['prev_activity'] = logon_events.groupby('user')['activity'].shift(1)
logon_events['prev_pc']       = logon_events.groupby('user')['pc'].shift(1)

suspicious_multipc = logon_events[
    (logon_events['activity'] == 'Logon') &
    (logon_events['prev_activity'] == 'Logon') &
    (logon_events['pc'] != logon_events['prev_pc'])
]

multi_pc_users = suspicious_multipc['user'].unique()

features_sessions['used_multi_pc_simultaneously'] = (
    features_sessions['user'].isin(multi_pc_users).astype(int)
)

features_sessions['session_duration_variability'] = (
    features_sessions['session_duration_variability'].fillna(0)
)


features_sessions.head()

,user,average_session_duration,session_duration_variability,nb_sessions,nb_short_sessions,ratio_short_sessions,used_multi_pc_simultaneously
0,AAB0162,11.061421,0.215556,251,0,0.0,0
1,AAB0398,9.092884,0.166152,356,0,0.0,0
2,AAC0610,9.100000,0.188562,2,0,0.0,0
3,AAC0668,9.080852,0.195071,356,0,0.0,0
4,AAC3270,8.080290,0.168880,356,0,0.0,0


In [27]:
logon_features = logon.groupby('user').agg(
    nb_logon = ('activity',lambda x : (x=='Logon').sum()),
    nb_logoff = ('activity',lambda x : (x=='Logoff').sum()),
    nb_pc_used = ('pc','nunique'),
    nb_pc_used_logon = ('pc',lambda x : x[logon.loc[x.index,'activity']=='Logon'].nunique()),
    nb_after_hours_logon = ('activity',lambda x : ((x=='Logon')&(logon.loc[x.index,'is_after_hours']==True)).sum()),
).reset_index()     

logon_features['unclosed_session_ratio'] = ((logon_features['nb_logon'] - logon_features['nb_logoff']).clip(lower=0) / logon_features['nb_logon']).fillna(0)
logon_features['after_hours_logon_ratio'] = (logon_features['nb_after_hours_logon'] / logon_features['nb_logon']).fillna(0)

weekly_logon_activity = logon_events.groupby(['user', 'week']).agg(
    nb_logon = ('activity','count'),
    nb_logon_after_hours = ('is_after_hours','sum') 
).reset_index()

logon_baseline = weekly_logon_activity.groupby('user').agg(
    baseline_median_logon = ('nb_logon','median'),
    baseline_median_logon_after_hours = ('nb_logon_after_hours','median'),
    baseline_std_logon = ('nb_logon','std'),
    baseline_std_logon_after_hours = ('nb_logon_after_hours','std'),
)

weekly_logon_activity = weekly_logon_activity.merge(logon_baseline,on='user',how='left')

weekly_logon_activity['zscore_logon'] = (
    (weekly_logon_activity['nb_logon'] - weekly_logon_activity['baseline_median_logon']) / weekly_logon_activity['baseline_std_logon'].clip(lower=1)
)

weekly_logon_activity['zscore_logon_after_hours'] = (
    (weekly_logon_activity['nb_logon_after_hours'] - weekly_logon_activity['baseline_median_logon_after_hours']) / weekly_logon_activity['baseline_std_logon_after_hours'].clip(lower=1)
)

zscore_logon_features = weekly_logon_activity.groupby('user').agg(
    max_zscore_logon      = ('zscore_logon', 'max'),
    max_zscore_logon_after_hours = ('zscore_logon_after_hours', 'max'),
).reset_index()

logon_features = logon_features.merge(zscore_logon_features, on='user', how='left')

logon_features.head()

,user,nb_logon,nb_logoff,nb_pc_used,nb_pc_used_logon,nb_after_hours_logon,unclosed_session_ratio,after_hours_logon_ratio,max_zscore_logon,max_zscore_logon_after_hours
0,AAB0162,355,356,1,1,0,0.000000,0.000000,0.00000,0.0
1,AAB0398,356,356,1,1,356,0.000000,1.000000,0.00000,0.0
2,AAC0610,665,387,1,1,28,0.418045,0.042105,2.02017,3.0
3,AAC0668,356,356,1,1,0,0.000000,0.000000,0.00000,0.0
4,AAC3270,356,356,1,1,0,0.000000,0.000000,0.00000,0.0


In [14]:
decoy_file = pd.read_parquet("cert_dataset/decoy_file.parquet")
decoy_file.head()

,decoy_filename,pc
0,C:\LJE2413\795JW126.jpg,PC-0302
1,C:\QMU9BC38.pdf,PC-6566
2,C:\GIS1668\YPS1RSIK.jpg,PC-2606
3,C:\KD02AETE.pdf,PC-5393
4,C:\AUZTDD4J.jpg,PC-8753


In [15]:
file = pd.read_parquet("cert_dataset/file.parquet")
file.head()

,id,date,user,pc,filename,activity,to_removable_media,from_removable_media,content
0,{F3E2-X3MV05YQ-3516SZDT},01/02/2010 07:19:41,SDH2394,PC-5849,R:\60WBQE7S.doc,File Open,False,True,"D0-CF-11-E0-A1-B1-1A-E1 Ernesztin's brother, L..."
1,{I6N1-Z7VL92UY-8715ESKQ},01/02/2010 07:21:30,SDH2394,PC-5849,R:\0VGILDW8.pdf,File Write,True,False,25-50-44-46-2D ---- Bengali As do many other T...
2,{G4X5-J7MH70FV-8936QVSB},01/02/2010 07:22:11,SDH2394,PC-5849,R:\60WBQE7S.doc,File Copy,False,True,"D0-CF-11-E0-A1-B1-1A-E1 Ernesztin's brother, L..."
3,{M2M7-Z5ST21EU-6704NSKO},01/02/2010 07:24:06,SDH2394,PC-5849,R:\22B5gX4\H8Y96RRE.doc,File Write,True,False,D0-CF-11-E0-A1-B1-1A-E1 After the death of his...
4,{R0A9-O9XB25PE-9236MALV},01/02/2010 07:24:45,SDH2394,PC-5849,R:\SDH2394\7XRCV2N5.pdf,File Copy,True,False,25-50-44-46-2D Although he restored some of th...


In [16]:
file_decoy = file[file['filename'].isin(decoy_file['decoy_filename'])]
file_decoy = file_decoy[['user','pc','filename']]
file_decoy.head()

,user,pc,filename
11,ABM3641,PC-7385,C:\77h89S6\DQSH4LBO.doc
15,AXW3243,PC-4929,C:\NTFU9CKZ.doc
21,HID1899,PC-2433,C:\386lN89\F868J5R7.pdf
22,DJA0740,PC-7197,C:\AS31ZQDZ.doc
25,ANC1950,PC-4921,C:\ANC1950\AIG3BQT7.pdf


In [17]:
file_decoy_features = file_decoy.groupby('user').agg(
    nb_access_decoy_file = ('filename','count'),
    nb_pc_decoy_file = ('pc','nunique')
).reset_index()

file_decoy_features.head()

,user,nb_access_decoy_file,nb_pc_decoy_file
0,AAB0162,30,1
1,AAB0398,36,1
2,AAC0610,191,1
3,AAC0668,62,1
4,AAC3270,8,1


### File Features

In [30]:
file = pd.read_parquet("cert_dataset/file.parquet")
file.head()

,id,date,user,pc,filename,activity,to_removable_media,from_removable_media,content
0,{F3E2-X3MV05YQ-3516SZDT},01/02/2010 07:19:41,SDH2394,PC-5849,R:\60WBQE7S.doc,File Open,False,True,"D0-CF-11-E0-A1-B1-1A-E1 Ernesztin's brother, L..."
1,{I6N1-Z7VL92UY-8715ESKQ},01/02/2010 07:21:30,SDH2394,PC-5849,R:\0VGILDW8.pdf,File Write,True,False,25-50-44-46-2D ---- Bengali As do many other T...
2,{G4X5-J7MH70FV-8936QVSB},01/02/2010 07:22:11,SDH2394,PC-5849,R:\60WBQE7S.doc,File Copy,False,True,"D0-CF-11-E0-A1-B1-1A-E1 Ernesztin's brother, L..."
3,{M2M7-Z5ST21EU-6704NSKO},01/02/2010 07:24:06,SDH2394,PC-5849,R:\22B5gX4\H8Y96RRE.doc,File Write,True,False,D0-CF-11-E0-A1-B1-1A-E1 After the death of his...
4,{R0A9-O9XB25PE-9236MALV},01/02/2010 07:24:45,SDH2394,PC-5849,R:\SDH2394\7XRCV2N5.pdf,File Copy,True,False,25-50-44-46-2D Although he restored some of th...


In [31]:
file['date'] = pd.to_datetime(file['date'], format='%m/%d/%Y %H:%M:%S')
file = file.drop(columns=['id'])
file = file.sort_values(by='date')

In [32]:
file['is_copy']   = file['activity'] == 'File Copy'
file['is_write']  = file['activity'] == 'File Write'
file['is_delete'] = file['activity'] == 'File Delete'
file['week'] = file['date'].dt.to_period('W')
file['to_rem']   = file['to_removable_media']
file['from_rem'] = file['from_removable_media']

file['is_weekend']    = file['date'].dt.dayofweek >= 5
file['is_after_hours'] = ~file['date'].dt.hour.between(7, 19)
file['is_off_hours']  = file['is_weekend'] | file['is_after_hours']

file = file.sort_values(['user', 'filename', 'date'])

file['prev_activity'] = file.groupby(['user', 'filename'])['activity'].shift(1)

file['open_then_copy']    = file['is_copy']   & (file['prev_activity'] == 'File Open')
file['write_then_delete'] = file['is_delete'] & (file['prev_activity'] == 'File Write')
file['copy_then_delete']  = file['is_delete'] & (file['prev_activity'] == 'File Copy')


file['copy_to_rem']  = file['is_copy']  & file['to_rem']
file['write_to_rem'] = file['is_write'] & file['to_rem']

In [33]:
SUSPICIOUS_KEYWORDS = [
    'confidential', 'secret', 'proprietary', 'classified', 'sensitive',
    'password', 'credential', 'private', 'restricted', 'internal',
    'top secret', 'do not share', 'not for distribution',
    'salary', 'payroll', 'ssn', 'social security',
    'patent', 'trade secret', 'intellectual property',
    'merger', 'acquisition', 'source code', 'api key', 'token', 'encryption key'
]


pattern = r'\b(?:' + '|'.join(map(re.escape, SUSPICIOUS_KEYWORDS)) + r')\b'


file['has_suspicious_content'] = file['content'].str.contains(
    pattern,
    case=False,
    na=False,
    regex=True
)

In [38]:
features_file = file.groupby('user').agg(
    nb_file_events        = ('date', 'count'),
    nb_file_copy          = ('is_copy', 'sum'),
    nb_file_write         = ('is_write', 'sum'),
    nb_file_delete        = ('is_delete', 'sum'),
    nb_copy_to_removable  = ('copy_to_rem', 'sum'),
    nb_write_to_removable = ('write_to_rem', 'sum'),
    nb_from_removable     = ('from_rem', 'sum'),
    nb_unique_files       = ('filename', 'nunique'),
    nb_pc_used_file       = ('pc', 'nunique'),
    nb_off_hours_file     = ('is_off_hours', 'sum'),
    nb_suspicious_content = ('has_suspicious_content', 'sum'),
).reset_index()

seq_features = file.groupby('user').agg(
    open_then_copy_count    = ('open_then_copy', 'sum'),
    write_then_delete_count = ('write_then_delete', 'sum'),
    copy_then_delete_count  = ('copy_then_delete', 'sum'),
).reset_index()

features_file = features_file.merge(seq_features, on='user', how='left')

d_events = features_file['nb_file_events'].clip(lower=1)
d_copy   = features_file['nb_file_copy'].clip(lower=1)
d_write  = features_file['nb_file_write'].clip(lower=1)
d_files  = features_file['nb_unique_files'].clip(lower=1)

features_file['copy_ratio']   = features_file['nb_file_copy']   / d_events
features_file['write_ratio']  = features_file['nb_file_write']  / d_events
features_file['delete_ratio'] = features_file['nb_file_delete'] / d_events

features_file['copy_to_removable_ratio']  = features_file['nb_copy_to_removable']  / d_copy
features_file['write_to_removable_ratio'] = features_file['nb_write_to_removable'] / d_write
features_file['from_removable_ratio']     = features_file['nb_from_removable']     / d_events

features_file['unique_files_ratio'] = features_file['nb_unique_files'] / d_events
features_file['events_per_file']    = features_file['nb_file_events']  / d_files
features_file['pc_usage_ratio']     = features_file['nb_pc_used_file'] / d_events

features_file['file_off_hours_ratio']          = features_file['nb_off_hours_file']     / d_events
features_file['suspicious_file_content_ratio'] = features_file['nb_suspicious_content'] / d_events

features_file['open_then_copy_ratio']    = features_file['open_then_copy_count']    / d_copy
features_file['write_then_delete_ratio'] = features_file['write_then_delete_count'] / d_write
features_file['copy_then_delete_ratio']  = features_file['copy_then_delete_count']  / d_copy

features_file = features_file.fillna(0).replace([float('inf'), -float('inf')], 0)

weekly_file_activity = file.groupby(['user', 'week']).agg(
    nb_file_activity = ('activity','count'),
).reset_index()

file_baseline = weekly_file_activity.groupby('user').agg(
    baseline_median_file = ('nb_file_activity','median'),
    baseline_std_file = ('nb_file_activity','std'),
)

weekly_file_activity = weekly_file_activity.merge(file_baseline,on='user',how='left')

weekly_file_activity['zscore_file_activity'] = (
    (weekly_file_activity['nb_file_activity'] - weekly_file_activity['baseline_median_file']) / weekly_file_activity['baseline_std_file'].clip(lower=1)
)
weekly_file_activity['zscore_file_activity'] = weekly_file_activity['zscore_file_activity'].fillna(0)

zscore_file_features = weekly_file_activity.groupby('user').agg(
    max_zscore_file_activity      = ('zscore_file_activity', 'max'),
).reset_index()

features_file = features_file.merge(zscore_file_features, on='user', how='left')


features_file = features_file[[
    'user', 'nb_file_events',
    'copy_ratio', 'write_ratio', 'delete_ratio',
    'copy_to_removable_ratio', 'write_to_removable_ratio', 'from_removable_ratio',
    'unique_files_ratio', 'events_per_file', 'pc_usage_ratio',
    'file_off_hours_ratio', 'suspicious_file_content_ratio',
    'open_then_copy_ratio', 'write_then_delete_ratio', 'copy_then_delete_ratio','max_zscore_file_activity'
]]


features_file.head()

,user,nb_file_events,copy_ratio,write_ratio,delete_ratio,copy_to_removable_ratio,write_to_removable_ratio,from_removable_ratio,unique_files_ratio,events_per_file,pc_usage_ratio,file_off_hours_ratio,suspicious_file_content_ratio,open_then_copy_ratio,write_then_delete_ratio,copy_then_delete_ratio,max_zscore_file_activity
0,AAB0162,30,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,0.266667,3.750000,0.033333,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
1,AAB0398,36,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,0.194444,5.142857,0.027778,0.027778,0.000000,0.000000,0.000000,0.000000,2.000000
2,AAC0610,1851,0.445705,0.126958,0.21826,0.447273,1.0,0.570502,0.346299,2.887676,0.000540,0.003241,0.036737,0.078788,0.251064,0.208485,2.022972
3,AAC0668,62,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,0.129032,7.750000,0.016129,0.000000,0.000000,0.000000,0.000000,0.000000,3.775115
4,AAC3270,8,0.000000,0.000000,0.00000,0.000000,0.0,0.000000,0.250000,4.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [41]:
dfs = [features_email,features_file,features_sessions,logon_features,file_decoy_features]

df_final = reduce(lambda left, right: pd.merge(left, right, on='user'), dfs)
df_final.to_csv('features_f.csv',index=False)